# Week 6 - Feature Engineering and Market Metrics
Engineer the key market indicators, add school districts using the properties longitude and latitute values, and perform segment analysis

In [1]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
from matplotlib import pyplot as plt
import seaborn as sns

## 1) Finish feature engineering

In [ ]:
# load in dataset
 
data_dir = Path("../data/cleaned_csv")

sold = pd.read_csv(
    data_dir / "sold_cleaned.csv",
    low_memory=False
)

listing = pd.read_csv(
    data_dir / 'listing_cleaned.csv',
    low_memory=False
)

print('Sold Shape:', sold.shape) # 413855, 91
print('Listing Shape:', listing.shape) # (546775, 81)

Sold Shape: (413855, 91)
Listing Shape: (546775, 81)


In [5]:
print(sold['LivingArea'].isna().sum())
print(sold['ClosePrice'].isna().sum())
print(sold['OriginalListPrice'].isna().sum())

234
2
756


In [8]:
sold_before = len(sold)

sold = sold[
    sold['ClosePrice'].notna() &
    (sold['ClosePrice'] > 0)
].copy()

print('Rows before:', sold_before)
print('Rows after:', len(sold))
print('Rows removed:', sold_before - len(sold))

Rows before: 413853
Rows after: 413853
Rows removed: 0


### Create key metrics from features
- Price Ratio - (ClosePrice / OriginalListPrice)
- Close to Original List Ratio - (ClosePrice / OriginalListPrice)

- Price Per Sq Ft - (ClosePrice / LivingArea)
- Days on Market - (DaysOnMarket, raw field)

In [ ]:
# keep valid values when calculating the price ratios
orig_price = sold["OriginalListPrice"].where(sold["OriginalListPrice"] > 0)
living_area = sold["LivingArea"].where(sold["LivingArea"] > 0)
close_price = sold["ClosePrice"].where(sold["ClosePrice"] > 0)

# create the key metrics
sold['price_ratio'] = close_price / orig_price
sold['price_per_sq_ft'] = close_price / living_area
sold["days_on_market"] = sold["DaysOnMarket"]

sold['close_to_original_list_ratio'] = close_price / orig_price # same formula as price ratio

sold[['ClosePrice', 'OriginalListPrice', 'LivingArea', 'price_ratio', 'price_per_sq_ft', 'days_on_market']].head(10)

,ClosePrice,OriginalListPrice,LivingArea,price_ratio,price_per_sq_ft,days_on_market
0,240000.0,499000.0,1140.0,0.480962,210.526316,777
1,815000.0,759900.0,1974.0,1.072510,412.867275,33
2,810000.0,739900.0,1974.0,1.094743,410.334347,228
3,2100000.0,2100000.0,3736.0,1.000000,562.098501,0
4,2340000.0,NaN,2442.0,NaN,958.230958,0
5,1485000.0,1550000.0,1601.0,0.958065,927.545284,0
6,1130000.0,999000.0,2136.0,1.131131,529.026217,1
7,1060000.0,1050000.0,1917.0,1.009524,552.947314,34
8,2150000.0,NaN,2200.0,NaN,977.272727,0
9,2000000.0,NaN,1690.0,NaN,1183.431953,0


In [9]:
# convert these to datetime because even though i did this earlier for some reason it's not datetime format ?????
date_fields = [
    'CloseDate',
    'PurchaseContractDate',
    'ListingContractDate',
    'ContractStatusChangeDate'
]

for column in date_fields:
    sold[column] = pd.to_datetime(
        sold[column],
        errors='coerce'
    )

    listing[column] = pd.to_datetime(
        listing[column],
        errors='coerce'
    )

In [10]:
# now closedate is datetime, so we can create the Year / Month / YrMo cols
sold['Year'] = sold['CloseDate'].dt.year
sold['Month'] = sold['CloseDate'].dt.month
sold['YrMo'] = sold['CloseDate'].dt.strftime('%Y-%m')


# create listing to contract days  & contract to close days
sold['listing_to_contract_days'] = (sold['PurchaseContractDate'] - sold['ListingContractDate']).dt.days
sold['contract_to_close_days'] = (sold['CloseDate'] - sold['PurchaseContractDate']).dt.days

sold[['Year', 'Month', 'YrMo', 'listing_to_contract_days', 'contract_to_close_days']]

,Year,Month,YrMo,listing_to_contract_days,contract_to_close_days
0,2024,1,2024-01,777.0,65.0
1,2024,1,2024-01,114.0,919.0
2,2024,1,2024-01,255.0,778.0
3,2024,1,2024-01,0.0,48.0
4,2024,1,2024-01,33.0,-33.0
...,...,...,...,...,...
413850,2026,4,2026-04,681.0,32.0
413851,2026,4,2026-04,703.0,32.0
413852,2026,4,2026-04,723.0,44.0
413853,2026,4,2026-04,691.0,80.0


In [24]:
# now do the same but for listing
listing['Year'] = listing['ListingContractDate'].dt.year
listing['Month'] = listing['ListingContractDate'].dt.month
listing['YrMo'] = listing['ListingContractDate'].dt.strftime('%Y-%m')

listing[['ListingContractDate', 'Year', 'Month', 'YrMo']].head(10)

,ListingContractDate,Year,Month,YrMo
0,2024-01-01,2024,1,2024-01
1,2024-01-24,2024,1,2024-01
2,2024-01-12,2024,1,2024-01
3,2024-01-20,2024,1,2024-01
4,2024-01-12,2024,1,2024-01
5,2024-01-30,2024,1,2024-01
6,2024-01-26,2024,1,2024-01
7,2024-01-08,2024,1,2024-01
8,2024-01-25,2024,1,2024-01
9,2024-01-29,2024,1,2024-01


In [12]:
feature_columns = [
    'price_ratio',
    'price_per_sq_ft',
    'days_on_market',
    'Year',
    'Month',
    'YrMo',
    'close_to_original_list_ratio',
    'listing_to_contract_days',
    'contract_to_close_days'
]

sold[feature_columns].head(15)

,price_ratio,price_per_sq_ft,days_on_market,Year,Month,YrMo,close_to_original_list_ratio,listing_to_contract_days,contract_to_close_days
0,0.480962,210.526316,777,2024,1,2024-01,0.480962,777.0,65.0
1,1.072510,412.867275,33,2024,1,2024-01,1.072510,114.0,919.0
2,1.094743,410.334347,228,2024,1,2024-01,1.094743,255.0,778.0
3,1.000000,562.098501,0,2024,1,2024-01,1.000000,0.0,48.0
4,NaN,958.230958,0,2024,1,2024-01,NaN,33.0,-33.0
5,0.958065,927.545284,0,2024,1,2024-01,0.958065,1.0,21.0
6,1.131131,529.026217,1,2024,1,2024-01,1.131131,1.0,17.0
7,1.009524,552.947314,34,2024,1,2024-01,1.009524,34.0,6.0
8,NaN,977.272727,0,2024,1,2024-01,NaN,5.0,-5.0
9,NaN,1183.431953,0,2024,1,2024-01,NaN,6.0,-6.0


In [15]:
feature_columns = [
    'price_ratio',
    'price_per_sq_ft',
    'days_on_market',
    'Year',
    'Month',
    'YrMo',
    'close_to_original_list_ratio',
    'listing_to_contract_days',
    'contract_to_close_days'
]

sold[feature_columns].head(15)

,price_ratio,price_per_sq_ft,days_on_market,Year,Month,YrMo,close_to_original_list_ratio,listing_to_contract_days,contract_to_close_days
0,0.480962,210.526316,777,2024,1,2024-01,0.480962,777.0,65.0
1,1.072510,412.867275,33,2024,1,2024-01,1.072510,114.0,919.0
2,1.094743,410.334347,228,2024,1,2024-01,1.094743,255.0,778.0
3,1.000000,562.098501,0,2024,1,2024-01,1.000000,0.0,48.0
4,NaN,958.230958,0,2024,1,2024-01,NaN,33.0,-33.0
5,0.958065,927.545284,0,2024,1,2024-01,0.958065,1.0,21.0
6,1.131131,529.026217,1,2024,1,2024-01,1.131131,1.0,17.0
7,1.009524,552.947314,34,2024,1,2024-01,1.009524,34.0,6.0
8,NaN,977.272727,0,2024,1,2024-01,NaN,5.0,-5.0
9,NaN,1183.431953,0,2024,1,2024-01,NaN,6.0,-6.0


In [13]:
# missing values for all the engineered metrics
missing_features = pd.DataFrame({
    'MissingCount': sold[feature_columns].isna().sum(),
    'MissingPercent': sold[feature_columns].isna().mean() * 100
})

missing_features = missing_features.sort_values(
    'MissingPercent',
    ascending=False
)

missing_features

,MissingCount,MissingPercent
price_ratio,758,0.183157
close_to_original_list_ratio,758,0.183157
price_per_sq_ft,234,0.056542
listing_to_contract_days,195,0.047118
contract_to_close_days,194,0.046877
days_on_market,0,0.000000
Year,0,0.000000
Month,0,0.000000
YrMo,0,0.000000


## 2) Add CA school districts
Use the latitude and longitude values and use the 2024-2025 district shapefile

In [14]:
# load california school district boundaries
district_path = Path(
    '../data/school_districts/DistrictAreas2425/DistrictAreas2425.shp'
)

school_districts = gpd.read_file(district_path)

school_districts.head()

,OBJECTID,Year,FedID,CDCode,CDSCode,CountyName,DistrictNa,DistrictTy,GradeLow,GradeHigh,...,MIGcount,MIGpct,SWDcount,SWDpct,SEDcount,SEDpct,DistrctAre,Shape__Are,Shape__Len,geometry
0,1,2024-25,0601770,0161119,01611190000000,Alameda,Alameda Unified,Unified,PK,12,...,0,0.0,1356,12.6,3958,36.7,11.248939,4.755489e+07,56522.982683,"MULTIPOLYGON (((-13606222.82 4540862.699, -136..."
1,2,2024-25,0601860,0161127,01611270000000,Alameda,Albany City Unified,Unified,PK,12,...,0,0.0,357,9.7,1184,32.1,1.789984,7.096327e+06,12696.382797,"POLYGON ((-13612893.866 4565099.707, -13612896..."
2,3,2024-25,0604740,0161143,01611430000000,Alameda,Berkeley Unified,Unified,PK,12,...,0,0.0,1111,12.1,2686,29.3,10.434329,4.364648e+07,43695.341538,"POLYGON ((-13609482.48 4565074.597, -13609483...."
3,4,2024-25,0607800,0161150,01611500000000,Alameda,Castro Valley Unified,Unified,PK,12,...,0,0.0,1113,11.6,3728,39.0,66.885571,2.838285e+08,142492.767565,"MULTIPOLYGON (((-13582508.535 4529067.071, -13..."
4,5,2024-25,0612630,0161168,01611680000000,Alameda,Emery Unified,Unified,PK,12,...,0,0.0,81,13.7,412,69.8,1.273929,5.363392e+06,13741.272894,"POLYGON ((-13613999.038 4555592.769, -13614126..."


In [18]:
print('District Dataset Shape - ', school_districts.shape)
school_districts.columns

District Dataset Shape -  (937, 54)


Index(['OBJECTID', 'Year', 'FedID', 'CDCode', 'CDSCode', 'CountyName',
       'DistrictNa', 'DistrictTy', 'GradeLow', 'GradeHigh', 'GradeLowCe',
       'GradeHighC', 'AssistStat', 'CongressUS', 'SenateCA', 'AssemblyCA',
       'UpdateNote', 'EnrollTota', 'EnrollChar', 'EnrollNonC', 'AAcount',
       'AApct', 'AIcount', 'AIpct', 'AScount', 'ASpct', 'FIcount', 'FIpct',
       'HIcount', 'HIpct', 'PIcount', 'PIpct', 'WHcount', 'WHpct', 'MRcount',
       'MRpct', 'NRcount', 'NRpct', 'ELcount', 'ELpct', 'FOScount', 'FOSpct',
       'HOMcount', 'HOMpct', 'MIGcount', 'MIGpct', 'SWDcount', 'SWDpct',
       'SEDcount', 'SEDpct', 'DistrctAre', 'Shape__Are', 'Shape__Len',
       'geometry'],
      dtype='str')

In [ ]:
# find the coordinate reference system (CRS) of the school districts dataset
school_districts.crs

<Projected CRS: EPSG:3857>
Name: WGS 84 / Pseudo-Mercator
Axis Info [cartesian]:
- X[east]: Easting (metre)
- Y[north]: Northing (metre)
Area of Use:
- name: World between 85.06°S and 85.06°N.
- bounds: (-180.0, -85.06, 180.0, 85.06)
Coordinate Operation:
- name: Popular Visualisation Pseudo-Mercator
- method: Popular Visualisation Pseudo Mercator
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [ ]:
# convert the boundary coordinates to standard lat/long (EPSG:4326)...
school_districts = school_districts.to_crs('EPSG:4326')

district_fields = [
    'CDCode',
    'CountyName',
    'DistrictNa',
    'DistrictTy',
    'geometry'
]

school_districts[district_fields]

,CDCode,CountyName,DistrictNa,DistrictTy,geometry
0,0161119,Alameda,Alameda Unified,Unified,"MULTIPOLYGON (((-122.22678 37.72651, -122.2267..."
1,0161127,Alameda,Albany City Unified,Unified,"POLYGON ((-122.28671 37.89852, -122.28673 37.8..."
2,0161143,Alameda,Berkeley Unified,Unified,"POLYGON ((-122.25606 37.89834, -122.25607 37.8..."
3,0161150,Alameda,Castro Valley Unified,Unified,"MULTIPOLYGON (((-122.01375 37.64265, -122.0114..."
4,0161168,Alameda,Emery Unified,Unified,"POLYGON ((-122.29663 37.8311, -122.29778 37.83..."
...,...,...,...,...,...
932,5872744,Yuba,Plumas Lake Elementary,Elementary,"POLYGON ((-121.59379 39.01033, -121.59351 39.0..."
933,5872751,Yuba,Wheatland,Elementary,"MULTIPOLYGON (((-121.46998 39.04551, -121.4692..."
934,5872769,Yuba,Wheatland Union High,High,"POLYGON ((-121.59379 39.01033, -121.59351 39.0..."
935,1964683,Los Angeles,Las Virgenes Unified,Unified,"POLYGON ((-118.64111 34.07482, -118.64126 34.0..."


In [ ]:
# check if we converted correctly
school_districts.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [23]:
school_districts['DistrictTy'].value_counts(dropna=False)

DistrictTy
Elementary    516
Unified       345
High           76
Name: count, dtype: int64

In [25]:
# keep only the district names, types, and geographic boundaries
district_lookup = school_districts[
    ['DistrictNa', 'DistrictTy', 'geometry']
].copy()

district_lookup.head()

,DistrictNa,DistrictTy,geometry
0,Alameda Unified,Unified,"MULTIPOLYGON (((-122.22678 37.72651, -122.2267..."
1,Albany City Unified,Unified,"POLYGON ((-122.28671 37.89852, -122.28673 37.8..."
2,Berkeley Unified,Unified,"POLYGON ((-122.25606 37.89834, -122.25607 37.8..."
3,Castro Valley Unified,Unified,"MULTIPOLYGON (((-122.01375 37.64265, -122.0114..."
4,Emery Unified,Unified,"POLYGON ((-122.29663 37.8311, -122.29778 37.83..."


In [27]:
# combine coordinates from sold and listing datasets for geographic matching
coordinates = pd.concat(
    [sold[['Latitude','Longitude']], listing[['Latitude', 'Longitude']]],
    ignore_index=True
)

# assign a unique identifier to each row
coordinates['property_row_id'] = coordinates.index


# make sure coordinates are numeric
coordinates['Latitude'] = pd.to_numeric(
    coordinates['Latitude'], errors='coerce'
)
coordinates['Longitude'] = pd.to_numeric(
    coordinates['Longitude'], errors='coerce'
)


# check for california coordinates

valid_coordinates = (
    coordinates['Latitude'].between(32, 42) &
    coordinates['Longitude'].between(-125, -114)
)

print('Total records:', len(coordinates))
print('Valid coordinates:', valid_coordinates.sum())
print('Invalid coordinates:', (~valid_coordinates).sum())

Total records: 960628
Valid coordinates: 875928
Invalid coordinates: 84700


In [ ]:
# now change only the valid latitude and longitude vals into geographic points 'GeoDataFrame' for spatial analysis
#turns the latitude/longitude pairs into geographic objects.
property_points = coordinates[valid_coordinates].copy()

property_points = gpd.GeoDataFrame(
    property_points,
    geometry=gpd.points_from_xy(
        property_points['Longitude'],
        property_points['Latitude']
    ),
    crs='EPSG:4326'
)

property_points.head()

,Latitude,Longitude,property_row_id,geometry
0,37.566106,-122.327954,0,POINT (-122.32795 37.56611)
1,32.659315,-117.096922,1,POINT (-117.09692 32.65932)
2,32.659284,-117.097174,2,POINT (-117.09717 32.65928)
3,35.277199,-120.646786,3,POINT (-120.64679 35.2772)
4,37.325454,-121.780303,4,POINT (-121.7803 37.32545)


In [29]:
# do spatial join to match properties with school districts w/ its coordinates

district_matches = gpd.sjoin(
    property_points,
    district_lookup,
    how='left',
    predicate='within'
)

print('Property points:', len(property_points))
print('District match rows:', len(district_matches))

district_matches[
    ['property_row_id', 'DistrictNa', 'DistrictTy']
].head(15)

Property points: 875928
District match rows: 1085507


,property_row_id,DistrictNa,DistrictTy
0,0,San Mateo-Foster City,Elementary
0,0,San Mateo Union High,High
1,1,National Elementary,Elementary
1,1,Sweetwater Union High,High
2,2,National Elementary,Elementary
2,2,Sweetwater Union High,High
3,3,San Luis Coastal Unified,Unified
4,4,East Side Union High,High
4,4,Evergreen Elementary,Elementary
5,5,San Diego Unified,Unified


Now that we've matched the properties to the school districts containing its coordinates, we can organize the districts into separate columns. This is because we want one property/row and have separate district columns for each elementary, unified, + high school districts.

In [49]:
# keep valid district matches and remove duplicate matches
district_clean = district_matches.dropna(subset=['DistrictNa', 'DistrictTy'])[[
    'property_row_id', 'DistrictNa', 'DistrictTy']].drop_duplicates()

# group district names by property and district type
district_grouped = (
    district_clean
    .groupby(['property_row_id', 'DistrictTy'])['DistrictNa']
    .apply(' | '.join)
    .unstack()
)

# create separate columns for elementary, high, and unified districts
district_wide = district_grouped.reindex(
    columns=['Elementary', 'High', 'Unified']
)

district_wide = district_wide.rename(
    columns={
        'Elementary': 'elementary_school_district',
        'High': 'high_school_district',
        'Unified': 'unified_school_district'
    }
)

district_wide.head(10)

DistrictTy,elementary_school_district,high_school_district,unified_school_district
property_row_id,,,
0,San Mateo-Foster City,San Mateo Union High,NaN
1,National Elementary,Sweetwater Union High,NaN
2,National Elementary,Sweetwater Union High,NaN
3,NaN,NaN,San Luis Coastal Unified
4,Evergreen Elementary,East Side Union High,NaN
5,NaN,NaN,San Diego Unified
6,NaN,NaN,Los Angeles Unified
7,NaN,NaN,Los Angeles Unified
8,Millbrae Elementary,San Mateo Union High,NaN


In [ ]:
# attach distrcits to sold and listing datasets
district_wide = district_wide.reindex(coordinates.index)

sold_districts = district_wide.iloc[:len(sold)].reset_index(drop=True)
listing_districts = district_wide.iloc[len(sold):].reset_index(drop=True)

# preserve the original number of rows
sold_rows = len(sold)
listing_rows = len(listing)

sold = sold.reset_index(drop=True)
listing = listing.reset_index(drop=True)

school_columns = district_wide.columns.tolist()

sold = sold.drop(columns=school_columns, errors='ignore')
listing = listing.drop(columns=school_columns, errors='ignore')

# add the new district information
sold = pd.concat([sold, sold_districts], axis=1)
listing = pd.concat([listing, listing_districts], axis=1)

# check whether a property matched any school district
sold['school_district_matched_flag'] = sold[school_columns].notna().any(axis=1)
listing['school_district_matched_flag'] = listing[school_columns].notna().any(axis=1)

print('Sold properties matched:', sold['school_district_matched_flag'].sum())
print('Listing properties matched:', listing['school_district_matched_flag'].sum())

Sold rows preserved: True
Listing rows preserved: True
Sold properties matched: 409541
Listing properties matched: 466082


In [44]:
sold[['City',
      'CountyOrParish',
      'Latitude',
      'Longitude',
      'elementary_school_district',
      'high_school_district',
      'unified_school_district']].head(15)

,City,CountyOrParish,Latitude,Longitude,elementary_school_district,high_school_district,unified_school_district
0,San Mateo,San Mateo,37.566106,-122.327954,San Mateo-Foster City,San Mateo Union High,NaN
1,National City,San Diego,32.659315,-117.096922,National Elementary,Sweetwater Union High,NaN
2,National City,San Diego,32.659284,-117.097174,National Elementary,Sweetwater Union High,NaN
3,San Luis Obispo,San Luis Obispo,35.277199,-120.646786,NaN,NaN,San Luis Coastal Unified
4,San Jose,Santa Clara,37.325454,-121.780303,Evergreen Elementary,East Side Union High,NaN
5,San Diego,San Diego,32.715089,-117.170568,NaN,NaN,San Diego Unified
6,Los Angeles,Los Angeles,34.070215,-118.258618,NaN,NaN,Los Angeles Unified
7,Los Angeles,Los Angeles,34.107850,-118.219273,NaN,NaN,Los Angeles Unified
8,Millbrae,San Mateo,37.589762,-122.399887,Millbrae Elementary,San Mateo Union High,NaN
9,San Mateo,San Mateo,37.535899,-122.282826,San Mateo-Foster City,San Mateo Union High,NaN


## 3) Perform segment analysis

In [43]:
# group by property type and subtype
property_groups = sold.groupby(
    ['PropertyType', 'PropertySubType'],
    dropna=False
)

property_counts = property_groups.size().rename('num_sales')
property_days = property_groups['days_on_market'].mean().rename('avg_days_on_market')

property_medians = property_groups[
    ['ClosePrice', 'price_per_sq_ft', 'price_ratio']
].median()

property_medians = property_medians.rename(columns={
    'ClosePrice': 'median_close_price',
    'price_per_sq_ft': 'median_ppsf',
    'price_ratio': 'median_price_ratio'
})

# combine all statistics into one table
property_summary = (
    property_medians
    .join(property_counts)
    .join(property_days)
    .reset_index()
    .sort_values('num_sales', ascending=False)
)

property_summary.head(10)

,PropertyType,PropertySubType,median_close_price,median_ppsf,median_price_ratio,num_sales,avg_days_on_market
14,Residential,SingleFamilyResidence,893450.0,532.724505,1.000000,309565,36.241700
3,Residential,Condominium,628000.0,565.106580,0.986133,68463,41.679126
18,Residential,Townhouse,805000.0,562.211982,1.000000,24285,32.062467
9,Residential,ManufacturedOnLand,324949.5,225.694444,0.969697,5340,58.654494
5,Residential,Duplex,915000.0,545.102380,0.983291,2263,42.506849
15,Residential,StockCooperative,360000.0,395.833333,0.985460,1663,37.913410
20,Residential,NaN,804000.0,570.308570,0.992494,804,44.748756
1,Residential,Cabin,238000.0,287.758241,0.912862,470,78.340426
19,Residential,Triplex,1105000.0,466.417910,0.972727,339,56.522124
10,Residential,MixedUse,700000.0,435.540070,0.933385,206,83.883495


In [45]:
# group by county and MLS area
geographic_groups = sold.groupby(
    ['CountyOrParish', 'MLSAreaMajor'],
    dropna=False
)

geographic_counts = geographic_groups.size().rename('num_sales')
geographic_days = geographic_groups['days_on_market'].mean().rename('avg_days_on_market')

geographic_medians = geographic_groups[
    ['ClosePrice', 'price_per_sq_ft', 'price_ratio']
].median()

geographic_medians = geographic_medians.rename(columns={
    'ClosePrice': 'median_close_price',
    'price_per_sq_ft': 'median_ppsf',
    'price_ratio': 'median_price_ratio'
})

geographic_summary = (
    geographic_medians
    .join(geographic_counts)
    .join(geographic_days)
    .reset_index()
    .sort_values('num_sales', ascending=False)
)

geographic_summary.head(10)

,CountyOrParish,MLSAreaMajor,median_close_price,median_ppsf,median_price_ratio,num_sales,avg_days_on_market
862,Riverside,SRCAR - Southwest Riverside County,585000.0,293.615200,0.996646,20183,41.684735
15,Contra Costa,NaN,825000.0,522.252498,1.000000,18159,28.637810
2,Alameda,NaN,1130000.0,698.428461,1.020408,17714,25.829457
1188,Santa Clara,699 - Not Defined,1610000.0,975.369458,1.026026,17127,21.401121
1152,San Mateo,699 - Not Defined,1732888.0,1059.259259,1.013639,6833,27.962242
804,Riverside,252 - Riverside,655745.0,379.335260,1.000000,5252,38.200305
8,Butte,NaN,399000.0,267.123288,0.981818,4015,51.701370
588,Monterey,699 - Not Defined,915000.0,593.462393,0.980095,3863,45.385193
800,Riverside,248 - Corona,755551.0,396.927882,0.995715,3325,38.977444
417,Los Angeles,LAC - Lancaster,475000.0,270.862355,1.000000,3069,41.190290


In [47]:
listing_office_groups = sold.groupby('ListOfficeName')

listing_office_summary = pd.DataFrame({
    'num_sales': listing_office_groups.size(),
    'total_sales_volume': listing_office_groups['ClosePrice'].sum(),
    'median_close_price': listing_office_groups['ClosePrice'].median()
}).reset_index()

listing_office_summary = listing_office_summary.sort_values(
    'total_sales_volume',
    ascending=False
)

listing_office_summary.head(10)

,ListOfficeName,num_sales,total_sales_volume,median_close_price
3976,Compass,29095,5.395475e+10,1350000.0
3888,Coldwell Banker Realty,18681,3.107472e+10,1195000.0
8967,Keller Williams Realty,8116,8.352135e+09,870000.0
1894,Berkshire Hathaway HomeServices California Pro...,5474,8.210250e+09,950000.0
7857,Intero Real Estate Services,4520,7.445594e+09,1375000.0
3920,Coldwell Banker West,2631,6.889841e+09,825000.0
5961,First Team Real Estate,5794,6.519418e+09,960000.0
16559,The Agency,2706,6.249440e+09,1670000.0
12220,Pacific Sotheby's Int'l Realty,1665,5.018710e+09,1650000.0
13727,Real Broker,4718,4.952605e+09,845000.0


In [48]:
buyer_office_groups = sold.groupby('BuyerOfficeName')

buyer_office_summary = pd.DataFrame({
    'num_sales': buyer_office_groups.size(),
    'total_sales_volume': buyer_office_groups['ClosePrice'].sum(),
    'median_close_price': buyer_office_groups['ClosePrice'].median()
}).reset_index()

buyer_office_summary = buyer_office_summary.sort_values(
    'total_sales_volume',
    ascending=False
)

buyer_office_summary.head(10)

,BuyerOfficeName,num_sales,total_sales_volume,median_close_price
4532,Compass,27168,4.934039e+10,1325000.0
4432,Coldwell Banker Realty,14954,2.565550e+10,1195000.0
10181,Keller Williams Realty,6368,8.079560e+09,835000.0
2181,Berkshire Hathaway HomeServices California Pro...,4255,6.338741e+09,975000.0
15703,Real Broker,6166,6.281231e+09,800000.0
8931,Intero Real Estate Services,3664,5.790139e+09,1337500.0
6818,First Team Real Estate,5238,5.713798e+09,900000.0
12737,NONMEMBER MRML,8835,5.633354e+09,503580.0
18810,The Agency,2532,5.601123e+09,1536560.0
21141,eXp Realty of California Inc,5278,5.509746e+09,835000.0


In [38]:
# check missing values in engineered features
missing_features = pd.DataFrame({
    'MissingCount': sold[feature_columns].isna().sum(),
    'MissingPercent': sold[feature_columns].isna().mean() * 100
})

# check invalid contract durations
print('Negative listing-to-contract days:',
      (sold['listing_to_contract_days'] < 0).sum())

print('Negative contract-to-close days:',
      (sold['contract_to_close_days'] < 0).sum())

Negative listing-to-contract days: 267
Negative contract-to-close days: 239


In [39]:
display(missing_features.sort_values('MissingPercent', ascending=False))

,MissingCount,MissingPercent
price_ratio,758,0.183157
close_to_original_list_ratio,758,0.183157
price_per_sq_ft,234,0.056542
listing_to_contract_days,195,0.047118
contract_to_close_days,194,0.046877
days_on_market,0,0.000000
Year,0,0.000000
Month,0,0.000000
YrMo,0,0.000000


In [40]:
# preview observations with all numeric features populated
sample_features = [
    'ListingId', 'ClosePrice', 'OriginalListPrice', 'LivingArea'
] + feature_columns + school_columns

display(
    sold.dropna(
        subset=[
            'price_ratio',
            'price_per_sq_ft',
            'listing_to_contract_days',
            'contract_to_close_days'
        ]
    )[sample_features].head(10)
)

,ListingId,ClosePrice,OriginalListPrice,LivingArea,price_ratio,price_per_sq_ft,days_on_market,Year,Month,YrMo,close_to_original_list_ratio,listing_to_contract_days,contract_to_close_days,elementary_school_district,high_school_district,unified_school_district
0,ML81865679,240000.0,499000.0,1140.0,0.480962,210.526316,777,2024,1,2024-01,0.480962,777.0,65.0,San Mateo-Foster City,San Mateo Union High,NaN
1,210014555,815000.0,759900.0,1974.0,1.072510,412.867275,33,2024,1,2024-01,1.072510,114.0,919.0,National Elementary,Sweetwater Union High,NaN
2,210008330,810000.0,739900.0,1974.0,1.094743,410.334347,228,2024,1,2024-01,1.094743,255.0,778.0,National Elementary,Sweetwater Union High,NaN
3,NS24001417,2100000.0,2100000.0,3736.0,1.000000,562.098501,0,2024,1,2024-01,1.000000,0.0,48.0,NaN,NaN,San Luis Coastal Unified
5,PTP2400188,1485000.0,1550000.0,1601.0,0.958065,927.545284,0,2024,1,2024-01,0.958065,1.0,21.0,NaN,NaN,San Diego Unified
6,P1-16615,1130000.0,999000.0,2136.0,1.131131,529.026217,1,2024,1,2024-01,1.131131,1.0,17.0,NaN,NaN,Los Angeles Unified
7,24356357,1060000.0,1050000.0,1917.0,1.009524,552.947314,34,2024,1,2024-01,1.009524,34.0,6.0,NaN,NaN,Los Angeles Unified
10,OC24025449,1939716.0,1939716.0,3383.0,1.000000,573.371564,0,2024,1,2024-01,1.000000,1.0,47.0,NaN,NaN,Saddleback Valley Unified
11,240002591SD,1080000.0,1080000.0,1224.0,1.000000,882.352941,0,2024,1,2024-01,1.000000,0.0,21.0,NaN,NaN,San Diego Unified
12,SR23229306,920000.0,899800.0,1963.0,1.022449,468.670402,0,2024,1,2024-01,1.022449,2.0,39.0,Castaic Union,William S. Hart Union High,NaN


In [42]:
print('Sold output:', sold.shape)
print('Listing output:', listing.shape)

Sold output: (413853, 104)
Listing output: (546775, 88)


## 4) Save the new datasets + reports after performing feature engineering

In [ ]:
# save feature-engineered datasets and reports
output_dir = Path('../data/feature_engineered')
report_dir = Path('../data/reports/week6_reports')

output_dir.mkdir(parents=True, exist_ok=True)
report_dir.mkdir(parents=True, exist_ok=True)

sold.to_csv(output_dir / 'sold_feature_engineered.csv', index=False)
listing.to_csv(output_dir / 'listings_feature_engineered.csv', index=False)

property_summary.to_csv(report_dir / 'property_summary.csv', index=False)
geographic_summary.to_csv(report_dir / 'geographic_summary.csv', index=False)
listing_office_summary.to_csv(report_dir / 'listing_office_summary.csv',index=False)
buyer_office_summary.to_csv(report_dir / 'buyer_office_summary.csv',index=False)
missing_features.to_csv(report_dir / 'missing_features.csv')

Sold output: (413853, 104)
Listing output: (546775, 88)
